## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-13-18-52-07 +0000,nyt,Europe Agrees on a Team to Represent It in Tal...,https://www.nytimes.com/2026/11/13/world/europ...
1,2026-11-13-18-50-00 +0000,wsj,Comex Gold Ends the Week 4.55% Lower at $3879.70,https://www.wsj.com/finance/commodities-future...
2,2026-11-13-18-46-00 +0000,wsj,Anthropic Chief Tells Jamie Dimon He Hopes IPO...,https://www.wsj.com/finance/anthropic-chief-te...
3,2026-11-13-18-42-58 +0000,nypost,Swiss dental startup vVardis shares slide 7% i...,https://nypost.com/2026/11/13/business/vvardis...
4,2026-11-13-18-42-53 +0000,nypost,‘Storm train’ barrels into Southern California...,https://nypost.com/2026/11/13/us-news/a-storm-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2473/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
142,nvidia,40
381,huang,35
380,jensen,27
99,new,25
154,trump,22


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
281,2026-11-13-00-44-00 +0000,wsj,Some of President Trump’s top advisers have st...,https://www.wsj.com/politics/policy/lutnick-fi...,159
248,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...,155
87,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...,148
405,2026-11-12-19-09-50 +0000,nypost,California Dem’s wife caught in curious Nvidia...,https://nypost.com/2026/11/12/us-news/rep-ro-k...,148
212,2026-11-13-10-00-00 +0000,wapo,How disgraced Nvidia CEO Jensen Huang became a...,https://www.washingtonpost.com/technology/inte...,138


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
281,159,2026-11-13-00-44-00 +0000,wsj,Some of President Trump’s top advisers have st...,https://www.wsj.com/politics/policy/lutnick-fi...
305,96,2026-11-12-23-05-02 +0000,missionlocal,"Richmond Buzz: New brunch, new pizza, and a ne...",https://missionlocal.org/2026/11/richmond-buzz...
248,93,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...
252,84,2026-11-13-03-27-00 +0000,wsj,"Sen. Chris Van Hollen (D., Md.) said he would ...",https://www.wsj.com/politics/policy/chris-van-...
87,69,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...
142,54,2026-11-13-13-19-19 +0000,chitri,"Ukraine fires nearly 1,200 drones at Russia as...",https://www.chicagotribune.com/2026/11/13/ukra...
121,48,2026-11-13-14-38-11 +0000,nyt,Kennedy’s Future: Will Democrats Try to Impeac...,https://www.nytimes.com/2026/11/12/us/politics...
112,45,2026-11-13-15-02-00 +0000,wsj,Of all the ways Chairman Kevin Warsh wants to ...,https://www.wsj.com/economy/central-banking/fe...
330,45,2026-11-12-22-02-00 +0000,wsj,Oil Futures Rebound as Trump Shrugs Off Need f...,https://www.wsj.com/finance/commodities-future...
224,43,2026-11-13-08-16-30 +0000,nypost,"Microsoft platforms including Outlook, Teams d...",https://nypost.com/2026/11/13/business/microso...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
